In [17]:
#Importing required libraries
import pandas as pd
import sqlite3 as sql

In [ ]:
# Define the path of the cleaned CSV file
csv_path = "data/cleaned_data.csv"

# Define the path where the SQLite database will be created
db_path = "database/books.db"

# Display the paths to verify they are correct
print("CSV path:", csv_path)
print("Database path:", db_path)

CSV path: data/cleaned_data.csv
Database path: database/books.db


In [3]:
#loading the dataset
df=pd.read_csv(csv_path)
df.head()

,title,category,price_gbp,rating,in_stock,price_inr
0,It's Only the Himalayas,Travel,45.17,2,True,4765.44
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,4,True,5214.86
2,See America: A Celebration of Our National Par...,Travel,48.87,3,True,5155.78
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,True,3897.17
4,Under the Tuscan Sun,Travel,37.33,3,True,3938.31


In [18]:
#Connecting to the SQLite database (it will create the database if it doesn't exist)
connect=sql.connect(db_path)
print("Database Successfully Connected to SQLite")
#Enable foreign key support
connect.execute("PRAGMA foreign_keys = ON;")
print("Foreign Key Support Enabled")

Database Successfully Connected to SQLite
Foreign Key Support Enabled


In [5]:
#create cursor object to execute SQL commands
cursor=connect.cursor()

#creating category table
cursor.execute('''create table category(
    category_id integer primary key,
    category_name text UNIQUE not null)
               ''')
connect.commit()
print("Category Table Created Successfully")

Category Table Created Successfully


In [6]:
# Get the unique category names from the DataFrame
unique_categories = df["category"].unique()

# Insert each unique category into the category table
for category in unique_categories:
    cursor.execute(
        """
        INSERT OR IGNORE INTO category(category_name)
        VALUES (?)
        """,
        (category,)
    )

# Save the inserted data
connect.commit()

# Display the categories that were inserted
print("Categories inserted successfully!")

Categories inserted successfully!


In [7]:
#query to verify Category table
query='''select * from category'''

result=pd.read_sql_query(query,connect)
print(result)

   category_id category_name
0            1        Travel
1            2       Mystery
2            3       Fiction
3            4       Science
4            5        Horror


In [8]:
# Create the books table with a primary key and foreign key
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    price_inr REAL NOT NULL,
    category_id INTEGER NOT NULL,

    FOREIGN KEY (category_id)
    REFERENCES category(category_id)
)
""")

# Save changes
connect.commit()

print("Books table created successfully!")

Books table created successfully!


In [9]:
# Read category IDs and category names from the category table
category_df = pd.read_sql_query(
    "SELECT category_id, category_name FROM category",
    connect
)

# Display the category DataFrame
category_df

,category_id,category_name
0,1,Travel
1,2,Mystery
2,3,Fiction
3,4,Science
4,5,Horror


In [10]:
# Merge the books DataFrame with the categories DataFrame
# This matches the category name with its corresponding category ID
books_df = df.merge(
    category_df,
    left_on="category",
    right_on="category_name",
    how="left"
)

# Display the first 5 rows
books_df.head()

,title,category,price_gbp,rating,in_stock,price_inr,category_id,category_name
0,It's Only the Himalayas,Travel,45.17,2,True,4765.44,1,Travel
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,Travel,49.43,4,True,5214.86,1,Travel
2,See America: A Celebration of Our National Par...,Travel,48.87,3,True,5155.78,1,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Travel,36.94,2,True,3897.17,1,Travel
4,Under the Tuscan Sun,Travel,37.33,3,True,3938.31,1,Travel


In [11]:
books_df=books_df[['title', 'price_gbp', 'rating', 'in_stock', 'price_inr', 'category_id']]

books_df.head()

,title,price_gbp,rating,in_stock,price_inr,category_id
0,It's Only the Himalayas,45.17,2,True,4765.44,1
1,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,4,True,5214.86,1
2,See America: A Celebration of Our National Par...,48.87,3,True,5155.78,1
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,True,3897.17,1
4,Under the Tuscan Sun,37.33,3,True,3938.31,1


In [12]:
# Insert all book records into the books table

# "append" adds the data to the existing books table
books_df.to_sql(
    "books",
    connect,
    if_exists="append",
    index=False
)

connect.commit()

print("Books inserted successfully!")

Books inserted successfully!


In [13]:
query='''select * from books'''
print(pd.read_sql_query(query,connect))

    book_id                                              title  price_gbp  \
0         1                            It's Only the Himalayas      45.17   
1         2  Full Moon over Noah’s Ark: An Odyssey to Mount...      49.43   
2         3  See America: A Celebration of Our National Par...      48.87   
3         4  Vagabonding: An Uncommon Guide to the Art of L...      36.94   
4         5                               Under the Tuscan Sun      37.33   
..      ...                                                ...        ...   
68       69                                             Misery      34.79   
69       70                                                 It      25.01   
70       71                                       'Salem's Lot      49.56   
71       72                                          The Stand      57.86   
72       73                        The Girl with All the Gifts      49.47   

    rating  in_stock  price_inr  category_id  
0        2         1    4765

In [14]:
# Count the total number of records in the books table
query = """
SELECT COUNT(*) AS total_books
FROM books
"""

# Display the result
pd.read_sql_query(query, connect)

,total_books
0,73


In [15]:
# SQL query to join books with their category names
query = """
SELECT 
    b.book_id,
    b.title,
    b.price_gbp,
    b.rating,
    b.in_stock,
    b.price_inr,
    c.category_name
FROM books AS b
JOIN category AS c
    ON b.category_id = c.category_id
"""

# Execute the query and store the result in a DataFrame
result = pd.read_sql_query(query, connect)

# Display the first 5 records
result.head()

,book_id,title,price_gbp,rating,in_stock,price_inr,category_name
0,1,It's Only the Himalayas,45.17,2,1,4765.44,Travel
1,2,Full Moon over Noah’s Ark: An Odyssey to Mount...,49.43,4,1,5214.86,Travel
2,3,See America: A Celebration of Our National Par...,48.87,3,1,5155.78,Travel
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,2,1,3897.17,Travel
4,5,Under the Tuscan Sun,37.33,3,1,3938.31,Travel


In [21]:
#close the cursor
cursor.close()

#close the connection to the database
connect.close()